# Benchmark modelos de odio en español (Colab)

Esta notebook aplica clasificadores HuggingFace de odio en español a un CSV ya subido al entorno.

**Antes de ejecutar:**
1. Subir manualmente el CSV en *Archivos* (panel izquierdo de Colab) o arrastrarlo
2. Seleccionar entorno GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*
3. Ajustar `INPUT_CSV` en la celda de configuración si el nombre de archivo difiere

## 1. Setup

In [1]:
import torch

device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if device == 0 else "CPU"
print(f"Dispositivo: {device_name}")

Dispositivo: Tesla T4


## 2. Instalación

Descomenta si hace falta.

In [2]:
!pip install -q pandas transformers torch tqdm scikit-learn accelerate

## 3. Imports

In [3]:
import re
from typing import Dict, Any, Optional

import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 4. Configuración

Ajusta estas variables según tu archivo.

In [5]:
INPUT_CSV   = 'tweets_dnu_clasificados.csv'  # nombre del CSV subido al entorno
OUTPUT_CSV  = 'DNU_modelos_hf.csv'

TEXT_COLUMN = 'text'   # columna con el texto a clasificar
GOLD_COLUMN = None     # columna con etiquetas de referencia; None si no hay gold
MAX_ROWS    = None     # None = procesar todo el archivo

## 5. Cargar datos

In [6]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f'Filas originales: {len(df)}')
print('Columnas disponibles:')
print(list(df.columns))
df.head(3)

Filas originales: 259752
Columnas disponibles:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels
0,Fri Mar 05 11:01:46 +0000 2021,1367792447334580225,1367792447334580225,RT @CELS_Argentina: ✅Celebramos la decisión de...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
1,Fri Mar 05 11:06:31 +0000 2021,1367793643239702529,1367793643239702529,RT @LANACION: Ley de Migraciones: el Gobierno ...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
2,Fri Mar 05 11:06:33 +0000 2021,1367793651192070144,1367793651192070144,"RT @CRLDEMONIO: Y lo peor , el alto mando d...","<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0


In [7]:
text_col = TEXT_COLUMN
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)
print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

Columna de texto: text
Filas a procesar: 259752


## 6. Modelos a comparar

- `cardiffnlp/twitter-xlm-roberta-base-hate-spanish` — orientado a hate speech en español en redes sociales.
- `JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish` — clasifica hate/offensive en español.
- `delarosajav95/HateSpeech-BETO-cased-v2` — BETO fine-tuneado para detección de odio en español.

Poner `False` en `USE_MODELS` para desactivar alguno.

In [8]:
USE_MODELS = {
    'cardiff_xlm_hate_es':     True,
    'bne_hate_offensive_es':   True,
    'beto_hate_v2_es':         True,
}

HF_MODEL_NAMES = {
    'cardiff_xlm_hate_es':   'cardiffnlp/twitter-xlm-roberta-base-hate-spanish',
    'bne_hate_offensive_es': 'JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish',
    'beto_hate_v2_es':       'delarosajav95/HateSpeech-BETO-cased-v2',
}

## 7. Cargar pipelines

In [9]:
loaded_models: Dict[str, Any] = {}
load_errors:   Dict[str, str] = {}

for key, model_name in HF_MODEL_NAMES.items():
    if USE_MODELS.get(key):
        try:
            loaded_models[key] = pipeline(
                'text-classification',
                model=model_name,
                tokenizer=model_name,
                truncation=True,
                device=device,
            )
            print(f'OK {key}: {model_name}')
        except Exception as e:
            load_errors[key] = str(e)
            print(f'ERROR {key}:', e)

print('\nErrores de carga:', load_errors)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/938 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-hate-spanish
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

OK beto_hate_v2_es: delarosajav95/HateSpeech-BETO-cased-v2

Errores de carga: {}


## 8. Funciones auxiliares

Convierte las etiquetas de cada modelo a un esquema binario común:
- `1` = odio / ofensivo
- `0` = no odio

In [10]:
NEGATIVE_TOKENS = {
    'non_hateful', 'non-hateful', 'not_hate', 'not-hate', 'no_hate', 'no-hate',
    'normal', 'neutral', 'none', 'clean', 'non-offensive', 'not offensive',
    'label_0', '0', 'neither',
}
POSITIVE_HINTS = {'hate', 'hateful', 'offensive', 'toxic', 'abusive', 'hs', 'aggressive', 'label_1', '1'}


def normalize_label_text(x: Any) -> str:
    if x is None:
        return ''
    s = str(x).strip().lower()
    return re.sub(r'\s+', ' ', s)


def binary_from_label(label: Any, score: Optional[float] = None) -> Optional[int]:
    s = normalize_label_text(label)
    if s in NEGATIVE_TOKENS:
        return 0
    for token in POSITIVE_HINTS:
        if token in s and s not in NEGATIVE_TOKENS:
            return 1
    return None


def run_hf_pipeline(model_key: str, text: str) -> Dict[str, Any]:
    clf = loaded_models[model_key]
    pred = clf(text)
    if isinstance(pred, list) and len(pred) > 0 and isinstance(pred[0], dict):
        best  = pred[0]
        label = best.get('label')
        score = best.get('score')
    else:
        label = str(pred)
        score = None
    return {'raw_label': label, 'score': score, 'binary_pred': binary_from_label(label, score)}

## 9. Prueba rápida con la primera fila

In [11]:
sample_text = str(work_df[text_col].iloc[0])
print(sample_text)

test_outputs = {}
for model_key in loaded_models:
    try:
        test_outputs[model_key] = run_hf_pipeline(model_key, sample_text)
    except Exception as e:
        test_outputs[model_key] = {'error': str(e)}

test_outputs

RT @CELS_Argentina: ✅Celebramos la decisión del Poder Ejecutivo de derogar el DNU 70/2017, cuya implementación significó un claro retroceso…


{'cardiff_xlm_hate_es': {'raw_label': 'NOT-HATE',
  'score': 0.9835401177406311,
  'binary_pred': 0},
 'bne_hate_offensive_es': {'raw_label': 'NEITHER',
  'score': 0.8745400905609131,
  'binary_pred': 0},
 'beto_hate_v2_es': {'raw_label': 'LABEL_0',
  'score': 0.9998912811279297,
  'binary_pred': 0}}

## 10. Correr todos los modelos

Usa batching nativo del pipeline. Bajar `BATCH_SIZE` a 32 si hay OOM.

In [12]:
BATCH_SIZE = 64

texts = work_df[text_col].fillna('').tolist()
n     = len(texts)

# Inicializar columnas
for model_key in loaded_models:
    for suffix in ['raw_label', 'score', 'binary_pred', 'error']:
        work_df[f'{model_key}__{suffix}'] = None

# Procesar cada modelo con pipeline batching
for model_key, clf in loaded_models.items():
    raw_labels, scores, binary_preds = [], [], []

    for out in tqdm(clf(texts, batch_size=BATCH_SIZE), total=n, desc=model_key):
        label = out.get('label')
        score = out.get('score')
        raw_labels.append(label)
        scores.append(score)
        binary_preds.append(binary_from_label(label, score))

    work_df[f'{model_key}__raw_label']   = raw_labels
    work_df[f'{model_key}__score']       = scores
    work_df[f'{model_key}__binary_pred'] = binary_preds

work_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nResultados guardados en {OUTPUT_CSV}')

cardiff_xlm_hate_es:   0%|          | 0/259752 [00:00<?, ?it/s]

bne_hate_offensive_es:   0%|          | 0/259752 [00:00<?, ?it/s]

beto_hate_v2_es:   0%|          | 0/259752 [00:00<?, ?it/s]


Resultados guardados en DNU_modelos_hf.csv


## 11. Vista rápida de resultados

In [13]:
cols_to_show = [text_col]
if GOLD_COLUMN is not None and GOLD_COLUMN in work_df.columns:
    cols_to_show.append(GOLD_COLUMN)
for model_key in loaded_models:
    cols_to_show += [f'{model_key}__raw_label', f'{model_key}__binary_pred']
work_df[cols_to_show].head(10)

,text,cardiff_xlm_hate_es__raw_label,cardiff_xlm_hate_es__binary_pred,bne_hate_offensive_es__raw_label,bne_hate_offensive_es__binary_pred,beto_hate_v2_es__raw_label,beto_hate_v2_es__binary_pred
0,RT @CELS_Argentina: ✅Celebramos la decisión de...,NOT-HATE,0,NEITHER,0,LABEL_0,0
1,RT @LANACION: Ley de Migraciones: el Gobierno ...,NOT-HATE,0,NEITHER,0,LABEL_0,0
2,"RT @CRLDEMONIO: Y lo peor , el alto mando d...",NOT-HATE,0,NEITHER,0,LABEL_0,0
3,RT @elcarpo: El Gobierno derogó el decreto de ...,NOT-HATE,0,NEITHER,0,LABEL_0,0
4,@joseantoniokast @sebastianpinera yo voté por ...,NOT-HATE,0,NEITHER,0,LABEL_0,0
5,RT @fargosi: El Gobierno derogó el decreto que...,NOT-HATE,0,NEITHER,0,LABEL_0,0
6,RT @VivianaaFerrari: Entienden que derogaron e...,NOT-HATE,0,NEITHER,0,LABEL_0,0
7,RT @LANACION: Ley de Migraciones: el Gobierno ...,NOT-HATE,0,NEITHER,0,LABEL_0,0
8,@todonoticias Derogó el DNU que impedía la ent...,NOT-HATE,0,NEITHER,0,LABEL_0,0
9,RT @LuisGasulla: Hacen todo mal.,NOT-HATE,0,OFFENSIVE,1,LABEL_0,0


## 12. Comparación entre modelos

Cuántos casos positivos marca cada uno.

In [14]:
summary_rows = []
for model_key in loaded_models:
    pred_col = f'{model_key}__binary_pred'
    valid = pd.to_numeric(work_df[pred_col], errors='coerce')
    summary_rows.append({
        'model':         model_key,
        'n_valid_preds': int(valid.notna().sum()),
        'n_pred_hate':   int((valid == 1).sum()),
        'hate_rate':     float((valid == 1).mean()) if valid.notna().any() else None,
    })
summary_df = pd.DataFrame(summary_rows).sort_values('hate_rate', ascending=False)
summary_df

,model,n_valid_preds,n_pred_hate,hate_rate
1,bne_hate_offensive_es,259752,76798,0.295659
0,cardiff_xlm_hate_es,259752,30856,0.118790
2,beto_hate_v2_es,259752,18130,0.069797


## 13. Acuerdo entre modelos

Correlación entre predicciones binarias.

In [15]:
pred_cols   = [f'{m}__binary_pred' for m in loaded_models]
pred_matrix = work_df[pred_cols].apply(pd.to_numeric, errors='coerce')
pred_matrix.corr()

,cardiff_xlm_hate_es__binary_pred,bne_hate_offensive_es__binary_pred,beto_hate_v2_es__binary_pred
cardiff_xlm_hate_es__binary_pred,1.000000,0.228056,0.160472
bne_hate_offensive_es__binary_pred,0.228056,1.000000,0.078028
beto_hate_v2_es__binary_pred,0.160472,0.078028,1.000000


## 14. Evaluación contra etiqueta de referencia (opcional)

Esta celda solo corre si `GOLD_COLUMN` está definida y existe en el CSV.

In [16]:
if GOLD_COLUMN is None or GOLD_COLUMN not in work_df.columns:
    print('GOLD_COLUMN no definida o no encontrada — salteando evaluación.')
else:
    gold_bin = pd.to_numeric(work_df[GOLD_COLUMN], errors='coerce')
    valid_gold = gold_bin.notna()
    print(f'Filas con gold label: {valid_gold.sum()}')

    for model_key in loaded_models:
        pred_col = f'{model_key}__binary_pred'
        preds    = pd.to_numeric(work_df[pred_col], errors='coerce')
        mask     = valid_gold & preds.notna()
        y_true   = gold_bin[mask].astype(int)
        y_pred   = preds[mask].astype(int)

        print('\n' + '=' * 60)
        print(f'Modelo: {model_key}')
        print(f'Predicciones válidas: {mask.sum()} / {len(work_df)}')
        print('=' * 60)
        if len(y_true) == 0:
            print('Sin predicciones válidas.')
            continue
        print('Accuracy:', round(accuracy_score(y_true, y_pred), 4))
        print('Confusion matrix:')
        print(confusion_matrix(y_true, y_pred))
        print()
        print(classification_report(y_true, y_pred, target_names=['NO ODIO', 'ODIO'], digits=4))

GOLD_COLUMN no definida o no encontrada — salteando evaluación.


## 15. Descargar resultado

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)